# MAMP-ml on Google Colab

Predict the immunogenicity of receptor–ligand pairs in plants using the pretrained ESM-2-based MAMP-ml model.

**This notebook walks through one complete workflow:**

1. Install `mamp-ml` from GitHub.
2. Upload (or otherwise stage) a spreadsheet of receptor–ligand pairs.
3. Install ColabFold and fold the receptors (one-time cost; takes ~5–15 min on a T4 GPU).
4. Run `mamp-ml predict` to drive the full pipeline plus model inference.
5. Download `predictions.csv` with per-row class probabilities.

> **Tip:** Pick *Runtime → Change runtime type → GPU* before running. ColabFold
> needs a GPU for fast folding; ESM-2 inference is faster too.

## 1. Install MAMP-ml

Installs from the `version2` branch on GitHub. The pip install fetches all Python dependencies (torch, transformers, fair-esm, biopython, scipy, etc.) and registers the `mamp-ml` console script.

In [ ]:
!pip install --quiet "git+https://github.com/DanielleMStevens/mamp-ml.git@version2"

In [ ]:
# Verify the install
!mamp-ml --help

## 2. Upload your input spreadsheet

The spreadsheet must have columns: `plant_species`, `receptor`, `locus_id`, `receptor_sequence`, `ligand_sequence`. See `example_data.xlsx` in the repo for the canonical layout.

In [ ]:
from google.colab import files
uploaded = files.upload()
input_filename = list(uploaded.keys())[0]
print(f'Using input: {input_filename}')

## 3. First pass: generate the receptor FASTA

`mamp-ml prepare` writes the receptor FASTA and then **checks whether ColabFold has been run**. Since this is your first run, it'll print the exact `colabfold_batch` command for you to run next and exit cleanly.

In [ ]:
!mamp-ml prepare "$input_filename"

## 4. Install ColabFold and fold the receptors

ColabFold needs its own Python environment to avoid clashing with `mamp-ml`'s torch / jax versions. The cells below install it in-process and run the fold.

In [ ]:
# Install ColabFold + a known-good JAX combination.
#
# Colab pre-installs jax >= 0.7.x along with the matching jax_cuda12_plugin,
# but `colabfold[alphafold]` downgrades jaxlib to 0.5.3, which leaves the
# pre-installed plugin incompatible (the runtime then silently falls back to
# CPU and folding takes hours). We pin the same versions as
# scripts/install_colabbatch_linux.sh — jax / jaxlib 0.4.35 with the matching
# CUDA12 plugin — for a working GPU stack.
!pip install --quiet "colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold"
!pip install --quiet "colabfold[alphafold]"
!pip uninstall -y -q jax jaxlib jax-cuda12-plugin jax-cuda12-pjrt jax-cuda12 jax-plugins
!pip install --quiet --upgrade "jax[cuda12]==0.4.35"
# Download the AF2 monomer ptm params (~3.6 GB).
!python -m colabfold.download AlphaFold2-ptm
# Sanity check: confirm jax sees the GPU through the matching CUDA plugin.
!python -c "import jax; print('jax', jax.__version__, '| devices:', jax.devices())"


In [ ]:
# Run ColabFold on the FASTA mamp-ml just produced. --num-models 1 --num-recycle 1
# keeps the wall-clock manageable; bump these if you want production-quality folds.
!colabfold_batch --num-models 1 --num-recycle 1 \
    intermediate_files/receptor_full_length.fasta \
    intermediate_files/receptor_only

## 5. Run prediction end-to-end

Now that ColabFold has produced PDB structures in `intermediate_files/receptor_only/`, `mamp-ml predict` will run **every remaining stage in one command**: structure post-processing, LRR-domain extraction, B-factor analysis, test-data assembly, chemical features, and ESM-2 inference.

In [ ]:
!mamp-ml predict "$input_filename" --device cuda

## 6. Inspect and download predictions

`predictions.csv` carries the original receptor/ligand metadata plus one column per predicted immunogenicity class.

In [ ]:
import pandas as pd
df = pd.read_csv('intermediate_files/predictions.csv')
df.head(10)

In [ ]:
from google.colab import files
files.download('intermediate_files/predictions.csv')

---

## Troubleshooting

* **`jax_cuda12_plugin version X is installed, but it is not compatible with the installed jaxlib`** — the install cell above pins `jax==0.4.35` and clears the orphaned CUDA12 plugin to fix exactly this. If you see this message *after* running that cell, rerun it; the `pip uninstall` line needs to land before the final `pip install` of `jax[cuda12]==0.4.35`.
* **ColabFold says "no GPU detected":** Make sure *Runtime → Change runtime type → GPU* is selected, then *Runtime → Restart and run all*.
* **`mamp-ml predict` reports "ColabFold has not been run yet":** Re-run the `colabfold_batch` cell and make sure it completes without errors. The `intermediate_files/receptor_only/log.txt` must be present.
* **HuggingFace download stalls:** ESM-2 weights are ~2.5 GB and downloaded on first inference. Retry; the cache persists across cells.
* **Want to re-use the same fold for a different ligand spreadsheet:** Keep `intermediate_files/receptor_only/` intact and only swap `input_filename`. `mamp-ml predict` will skip the fold step automatically.

## Power-user escape hatches

`mamp-ml` exposes one subcommand per pipeline stage if you want to inspect intermediates:

* `mamp-ml prepare-fasta <xlsx> <out_fasta>` — just stage 1
* `mamp-ml structure-stage <colabfold_dir> <scores.txt> <pdb_dir> <lrr.txt>` — structure post-processing
* `mamp-ml lrr-domain-fasta <lrr.txt> <receptor_fasta> <out_fasta>` — LRR-domain extraction
* `mamp-ml bfactor <pdb_dir> <cache_dir> <out_csv>` — B-factor bandpass
* `mamp-ml assemble-test-data <xlsx> <lrr_fasta> <out_csv>` — per-row assembly
* `mamp-ml chemical-features <in_csv> <out_csv>` — chemical features
